# 11) Assignment Breakdown Agent

This notebook builds a very small, engaging Python "agent" using the GAME structure:

- G — Goals: what success looks like and simple rules
- A — Actions: the tools the agent can call (plain Python functions)
- M — Memory: small persistent state (saved to a JSON file)
- E — Environment: executes tools and returns feedback to the agent loop

Learning objective: Students understand the agent loop (decide -> act -> observe -> repeat) with minimal coding.

Tip for class demos: If the agent uses waiting/timers, enable DEMO_MODE=True to shorten waits.


## G — Goals

Reduce overwhelm: read an assignment and output only the next 10-minute step. Save plans.

## Simple rules

- Ask for assignment text.
- Output one next step.
- Save to memory.

## How to run

1. Run all cells top-to-bottom.
2. When the agent loop starts, interact in the notebook input prompts.
3. Stop anytime by typing: `stop` (or `exit`, `quit`).


In [ ]:
from __future__ import annotations

import json, os, random, time
from dataclasses import dataclass
from typing import Any, Dict, List, Optional

def should_stop(text: str) -> bool:
    return text.strip().lower() in {"stop", "exit", "quit", "end"}

@dataclass
class Action:
    # Structured action output: tool_name + args
    tool_name: str
    args: Dict[str, Any]

class Tools:
    # A: Actions (Tools) - tiny functions the agent can call
    @staticmethod
    def notify(message: str) -> str:
        print(f"\n[Agent] {message}")
        return "ok"

    @staticmethod
    def ask(prompt: str) -> str:
        return input(f"\n[You] {prompt}\n> ").strip()

    @staticmethod
    def load_memory(path: str) -> Dict[str, Any]:
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)
        return {}

    @staticmethod
    def save_memory(path: str, data: Dict[str, Any]) -> str:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
        return "saved"

class Environment:
    # E: Environment - runs tools and returns feedback + updated memory
    def __init__(self, memory_path: str):
        self.memory_path = memory_path

    def execute(self, action: Action, memory: Dict[str, Any]) -> Dict[str, Any]:
        if action.tool_name == "notify":
            Tools.notify(str(action.args.get("message", "")))
            return {"result": "notified", "memory": memory}

        if action.tool_name == "ask":
            ans = Tools.ask(str(action.args.get("prompt", "")))
            return {"result": ans, "memory": memory}

        if action.tool_name == "load_memory":
            loaded = Tools.load_memory(self.memory_path)
            return {"result": "loaded", "memory": loaded}

        if action.tool_name == "save_memory":
            Tools.save_memory(self.memory_path, memory)
            return {"result": "saved", "memory": memory}

        if action.tool_name == "terminate":
            return {"result": "terminated", "memory": memory}

        return {"result": "unknown_tool", "memory": memory}


## Implementation (GAME Agent Loop)
The code below is intentionally small and readable.

In [ ]:
MEMORY_FILE = "11_assignment_breakdown_memory.json"

NEXT_STEPS = [
    "Rewrite the problem in your own words (3 lines).",
    "List inputs/outputs and 2 example test cases.",
    "Make a TODO list of 4 small tasks.",
    "Sketch a simple algorithm in 5 bullets.",
    "Identify the smallest subtask you can code first.",
]

def choose_step(text: str) -> str:
    t = text.lower()
    if "implement" in t or "code" in t:
        return "Make a TODO list of 4 small tasks."
    if "analyze" in t or "compare" in t:
        return "Rewrite the problem in your own words (3 lines)."
    return random.choice(NEXT_STEPS)

def run_agent():
    env = Environment(MEMORY_FILE)
    memory = env.execute(Action("load_memory", {}), {})["memory"] or {}
    plans = memory.get("plans", [])

    Tools.notify("Assignment Breakdown Agent is running. Type 'stop' to end.")

    while True:
        txt = Tools.ask("Paste assignment statement (short):")
        if should_stop(txt):
            break
        step = choose_step(txt)
        Tools.notify(f"Next 10-minute step:\n- {step}")

        plans.append({"assignment": txt, "next_step": step, "ts": time.time()})
        memory["plans"] = plans[-50:]
        env.execute(Action("save_memory", {}), memory)

        again = Tools.ask("Another assignment? (yes/no)")
        if again.lower().startswith("n"):
            break

    env.execute(Action("save_memory", {}), memory)
    Tools.notify("Saved. Bye!")

run_agent()


## Easy extensions

- Ask available time and adapt.
- Export to Markdown.
- Review last plan.